# 🧠📚 Job Interview Guide — Personalized Study Notebook
**Student:** MinhThuan  
**Date:** 2026-02-27  
**Quiz Score:** 2 / 15 (13%)  
**Model Used:** Claude Sonnet 4.6

---

This notebook targets the topics you missed in the interview quiz.  
Each section includes a conceptual explanation, worked example, and a hands-on exercise with TODOs.

### Topics Covered
1. Supervised vs. Unsupervised Learning
2. Dependent vs. Independent Variables
3. Data Leakage & Train / Validation / Test Split
4. Logistic Regression — Intercept, Slope, and Cross-Entropy
5. KNN — Hyperparameters, Overfitting, and Curse of Dimensionality
6. Decision Trees — Leaf Nodes, Predictions, and max_depth
7. Class Imbalance — Why Accuracy Lies
8. ML Pipeline Pattern — Preventing Leakage in Cross-Validation
9. Reflection

---
## 1. Supervised vs. Unsupervised Learning

### Concept

| | Supervised | Unsupervised |
|---|---|---|
| **Labels** | Required (input → output pairs) | Not required |
| **Goal** | Learn a mapping from X to y | Find hidden structure in X |
| **Examples** | Linear regression, Logistic regression, KNN, Decision Trees | K-Means clustering, PCA, Autoencoders |
| **Output** | Predicted label or value | Cluster assignment, embedding, compressed representation |

> ⚠️ **Common confusion:** Reinforcement Learning (RL) is a *third* paradigm — an agent learns by receiving rewards/penalties through trial and error. It is **not** supervised or unsupervised.

### Key rule of thumb
- If you have a **target column (y)** you are trying to predict → **Supervised**
- If you are exploring structure **without a target** → **Unsupervised**

In [ ]:
# Supervised example: predict species label (y exists)
from sklearn.datasets import load_iris
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clf = KNeighborsClassifier(n_neighbors=3)
clf.fit(X_train, y_train)
print('Supervised (KNN) accuracy:', accuracy_score(y_test, clf.predict(X_test)))

# Unsupervised example: find clusters (no y used)
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, random_state=42, n_init='auto')
kmeans.fit(X)  # no y passed
print('Unsupervised (KMeans) cluster labels (first 10):', kmeans.labels_[:10])

### ✏️ Exercise 1
In the cell below, answer in comments:
1. Which paradigm is used for spam email detection? Why?
2. Which paradigm is used for customer segmentation when you have no purchase categories? Why?
3. Give one real-world example of reinforcement learning.

In [ ]:
# TODO: Answer the three questions as comments below

# 1. Spam detection:

# 2. Customer segmentation (no categories):

# 3. Real-world RL example:


---
## 2. Dependent vs. Independent Variables

### Concept

| Term | ML Equivalent | Description |
|---|---|---|
| **Independent variable** | Feature / Input / X | The variable(s) you use to make predictions |
| **Dependent variable** | Target / Label / y | The variable you are trying to predict — it *depends* on the inputs |

### Examples
| Scenario | Independent (X) | Dependent (y) |
|---|---|---|
| Predict house price | Size, location, bedrooms | Price |
| Predict student pass/fail | Hours studied, attendance | Pass (1) or Fail (0) |
| Predict blood pressure | Age, weight, diet | Blood pressure reading |

> ⚠️ **ID columns** (student ID, row number) are **neither** — they carry no predictive signal and must be dropped.

In [ ]:
import pandas as pd

df = pd.DataFrame({
    'student_id': [101, 102, 103, 104, 105],
    'hours_studied': [2, 5, 1, 8, 3],
    'attendance_pct': [70, 90, 50, 95, 75],
    'exam_score': [55, 82, 40, 95, 60]   # <-- dependent variable
})

# Correct: drop ID, separate X and y
X = df[['hours_studied', 'attendance_pct']]  # independent variables
y = df['exam_score']                          # dependent variable

print('Features (X):\n', X)
print('\nTarget (y):\n', y)

### ✏️ Exercise 2
Given the dataset below, identify and separate the correct X and y. Remove any columns that should not be used.

In [ ]:
import pandas as pd

raw = pd.DataFrame({
    'patient_id':     [1, 2, 3, 4, 5],
    'age':            [45, 62, 38, 55, 70],
    'weight_kg':      [80, 95, 65, 88, 102],
    'smoker':         [1, 0, 0, 1, 1],
    'blood_pressure': [130, 155, 115, 145, 160]  # target
})

# TODO: Create X (features) and y (target). Drop non-predictive columns.
X = None  # replace with correct columns
y = None  # replace with target column

print('X:', X)
print('y:', y)

---
## 3. Data Leakage & Train / Validation / Test Split

### Concept

**Data leakage** occurs when information from outside the training set is used to build the model, causing optimistically biased evaluation.

**The golden rule:** Any preprocessing step that *learns* from data (e.g., computing mean/std for scaling, fitting an imputer) must be **fit only on training data**, then applied to validation and test sets.

### Three-way split
```
Full Dataset
├── Train set   (~70%)  → fit model + preprocessors
├── Validation  (~15%)  → tune hyperparameters, early stopping
└── Test set    (~15%)  → final unbiased evaluation (touch once!)
```

### Why stratify?
Stratification ensures each split has the **same class proportions** as the full dataset — critical for imbalanced data.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X, y = load_breast_cancer(return_X_y=True)

# ✅ CORRECT: 3-way stratified split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)
print(f'Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')

# Fit scaler ONLY on train
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)    # transform only — no fit!
X_test_s  = scaler.transform(X_test)   # transform only — no fit!

model = LogisticRegression(max_iter=1000)
model.fit(X_train_s, y_train)
print('Val accuracy  (correct):', accuracy_score(y_val, model.predict(X_val_s)))
print('Test accuracy (correct):', accuracy_score(y_test, model.predict(X_test_s)))

print()

# ❌ WRONG: scaler fit on full data before split (leakage)
scaler_bad = StandardScaler().fit(X)       # sees val and test data!
X_bad = scaler_bad.transform(X)
X_tr_bad, X_te_bad, y_tr_bad, y_te_bad = train_test_split(
    X_bad, y, test_size=0.20, random_state=42
)
model_bad = LogisticRegression(max_iter=1000).fit(X_tr_bad, y_tr_bad)
print('Test accuracy (LEAKAGE):', accuracy_score(y_te_bad, model_bad.predict(X_te_bad)),
      '<-- inflated!')

### ✏️ Exercise 3
The code below has a data leakage bug. Find it, fix it, and explain what was wrong.

In [ ]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

X, y = load_wine(return_X_y=True)

# --- BUGGY CODE: find and fix the leakage ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)   # BUG IS HERE

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y, random_state=0
)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
print('Accuracy:', accuracy_score(y_test, knn.predict(X_test)))

# TODO: Rewrite correctly (fit scaler only on X_train)
# TODO: Add a comment explaining why the original code was wrong


---
## 4. Logistic Regression — Intercept, Slope, and Cross-Entropy

### Concept

Logistic regression models the **log-odds** of the positive class:

$$z = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \ldots$$

$$P(y=1) = \sigma(z) = \frac{1}{1+e^{-z}}$$

| Parameter | Name | Meaning |
|---|---|---|
| **β₀** | Intercept | Log-odds when **all features = 0**; shifts the sigmoid left/right |
| **β₁, β₂, …** | Coefficients (slopes) | Change in log-odds per unit increase in each feature |

> ⚠️ β₀ is the **log-odds**, NOT the probability directly. To get probability: `σ(β₀) = 1 / (1 + exp(-β₀))`.

### Cross-Entropy Loss

$$\text{Loss} = -\frac{1}{N}\sum_{i=1}^{N}\left[y_i \log(p_i) + (1-y_i)\log(1-p_i)\right]$$

| True label | Prediction | Formula used | Loss |
|---|---|---|---|
| y=1 | p=0.95 | -log(0.95) | Low ✅ |
| y=1 | p=0.10 | -log(0.10) | High ❌ |
| y=0 | p=0.05 | -log(1-0.05)=-log(0.95) | Low ✅ |
| y=0 | p=0.95 | -log(1-0.95)=-log(0.05) | **Very high** ❌ |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
from sklearn.metrics import log_loss

X, y = make_classification(n_samples=300, n_features=1, n_informative=1,
                            n_redundant=0, n_clusters_per_class=1, random_state=7)

model = LogisticRegression()
model.fit(X, y)

print(f'Intercept (β₀): {model.intercept_[0]:.4f}')
print(f'  → Log-odds when X=0: {model.intercept_[0]:.4f}')
print(f'  → Probability when X=0: {1/(1+np.exp(-model.intercept_[0])):.4f}')
print(f'Coefficient (β₁): {model.coef_[0][0]:.4f}')
print(f'  → Each +1 unit increase in X changes log-odds by {model.coef_[0][0]:.4f}')

proba = model.predict_proba(X)[:, 1]
print(f'\nCross-entropy loss: {log_loss(y, proba):.4f}')

x_range = np.linspace(X.min(), X.max(), 300).reshape(-1, 1)
p_range = model.predict_proba(x_range)[:, 1]

plt.figure(figsize=(9, 4))
plt.scatter(X, y, alpha=0.4, label='Data')
plt.plot(x_range, p_range, linewidth=2, label='Sigmoid P(y=1)')
plt.axhline(0.5, linestyle='--', color='gray', label='Threshold 0.5')
plt.xlabel('Feature')
plt.ylabel('P(y=1)')
plt.title('Logistic Regression — Sigmoid Curve')
plt.legend()
plt.show()

In [ ]:
# Cross-entropy sanity check — compute manually for 4 cases
import numpy as np

cases = [
    {'y': 1, 'p': 0.95, 'label': 'Correct & confident  (y=1, p=0.95)'},
    {'y': 1, 'p': 0.10, 'label': 'Wrong & confident    (y=1, p=0.10)'},
    {'y': 0, 'p': 0.05, 'label': 'Correct & confident  (y=0, p=0.05)'},
    {'y': 0, 'p': 0.95, 'label': 'Wrong & confident    (y=0, p=0.95)'},
]

for c in cases:
    y_val, p = c['y'], c['p']
    loss = -(y_val * np.log(p) + (1 - y_val) * np.log(1 - p))
    print(f"{c['label']:45s}  loss = {loss:.4f}")

### ✏️ Exercise 4
Complete the function below and verify it against `sklearn.metrics.log_loss`.

In [ ]:
import numpy as np
from sklearn.metrics import log_loss as sklearn_log_loss

def my_log_loss(y_true, y_prob):
    """
    Compute binary cross-entropy loss.
    Args:
        y_true: array of true labels (0 or 1)
        y_prob: array of predicted probabilities for class 1
    Returns:
        float: average log loss
    """
    eps = 1e-15
    y_prob = np.clip(y_prob, eps, 1 - eps)
    # TODO: implement the formula and return the average loss
    pass

# Sanity check
y_true = np.array([1, 1, 0, 0])
y_prob = np.array([0.9, 0.8, 0.1, 0.2])

my_result  = my_log_loss(y_true, y_prob)
sk_result  = sklearn_log_loss(y_true, y_prob)

print(f'My log loss:      {my_result}')
print(f'Sklearn log loss: {sk_result:.6f}')
# TODO: confirm they match

---
## 5. KNN — Hyperparameters, Overfitting, and Curse of Dimensionality

### Concept

**KNN hyperparameters** (set before training — not learned from data):

| Hyperparameter | Effect |
|---|---|
| `n_neighbors` (k) | Low k → complex boundary, high variance (overfitting). High k → smooth boundary, high bias (underfitting) |
| `metric` | Distance function: `euclidean`, `manhattan`, `minkowski` |
| `weights` | `uniform` (all neighbors equal) vs `distance` (closer neighbors weighted more) |
| `p` | Minkowski power: p=1 → Manhattan, p=2 → Euclidean |

> ⚠️ KNN has **no learned parameters** (no weights, no coefficients). It memorizes training data. It is **non-parametric**.

### k=1 Overfitting
With k=1, every training point is its own nearest neighbor → 100% train accuracy. Any noise is memorized.

### Curse of Dimensionality
In high-dimensional spaces, all pairwise distances converge to similar values. "Nearest" neighbors are no longer meaningfully close → KNN breaks down. Fix: apply PCA or feature selection before KNN.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

ks = range(1, 31)
train_scores, test_scores = [], []

for k in ks:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_s, y_train)
    train_scores.append(knn.score(X_train_s, y_train))
    test_scores.append(knn.score(X_test_s, y_test))

plt.figure(figsize=(10, 4))
plt.plot(list(ks), train_scores, marker='o', label='Train accuracy')
plt.plot(list(ks), test_scores,  marker='s', label='Test accuracy')
plt.axvline(1, linestyle='--', color='red', alpha=0.5, label='k=1 (overfit)')
plt.xlabel('k (n_neighbors)')
plt.ylabel('Accuracy')
plt.title('KNN: Train vs Test Accuracy — Overfitting at Low k')
plt.legend()
plt.show()

best_k = list(ks)[test_scores.index(max(test_scores))]
print(f'Best k: {best_k}  |  Test accuracy: {max(test_scores):.4f}')
print(f'k=1 train acc: {train_scores[0]:.4f}  |  k=1 test acc: {test_scores[0]:.4f}  <- overfitting')

In [ ]:
# Curse of dimensionality demo
import numpy as np

np.random.seed(0)
n_samples = 500

print(f'{'Dimensions':>12}  |  Mean NN distance  |  Std')
print('-' * 50)

for n_dims in [2, 10, 50, 100, 500]:
    data = np.random.randn(n_samples, n_dims)
    dists = np.linalg.norm(data[1:] - data[0], axis=1)
    print(f'{n_dims:>12}  |  {dists.mean():>18.4f}  |  {dists.std():.4f}')

print()
print('As dimensions grow, mean distance increases but std SHRINKS relative to mean.')
print('All points become roughly equidistant -> nearest neighbor is meaningless.')

### ✏️ Exercise 5
Using the Iris dataset, sweep k from 1 to 20 using `metric='manhattan'`. Print each k and its test accuracy, then print the best k.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

X, y = load_iris(return_X_y=True)

# TODO: Split (stratified), scale, sweep k=1..20 with metric='manhattan'
# TODO: Print each k and its test accuracy
# TODO: Print the best k


---
## 6. Decision Trees — Leaf Nodes, Predictions, and max_depth

### Concept

A Decision Tree recursively splits data based on feature thresholds:

```
Root Node
├── [feature <= threshold]  ← Internal node (split)
│   ├── Internal Node
│   │   ├── Leaf Node → predict class A   ← terminal, no more splits
│   │   └── Leaf Node → predict class B
│   └── Leaf Node → predict class C
└── [feature > threshold]
    └── Leaf Node → predict class A
```

**Leaf node:** Terminal node. Outputs:
- **Classification:** majority class of training samples that reached this leaf
- **Regression:** mean value of training samples that reached this leaf

### Overfitting with max_depth
| `max_depth` | Effect |
|---|---|
| `None` (unlimited) | Grows until all leaves are pure → 100% train accuracy, overfits noise |
| Small (e.g., 3–5) | Constrains tree → lower variance, better generalization |

In [ ]:
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import numpy as np

X, y = load_iris(return_X_y=True)
feature_names = ['sepal_len', 'sepal_wid', 'petal_len', 'petal_wid']
class_names   = ['setosa', 'versicolor', 'virginica']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=0
)

# Compare unlimited vs constrained depth
for max_d in [None, 3]:
    dt = DecisionTreeClassifier(max_depth=max_d, random_state=0)
    dt.fit(X_train, y_train)
    print(f'max_depth={str(max_d):>4}  '
          f'train={accuracy_score(y_train, dt.predict(X_train)):.3f}  '
          f'test={accuracy_score(y_test, dt.predict(X_test)):.3f}  '
          f'n_leaves={dt.get_n_leaves()}')

# Visualize constrained tree
dt3 = DecisionTreeClassifier(max_depth=3, random_state=0)
dt3.fit(X_train, y_train)

plt.figure(figsize=(14, 6))
plot_tree(dt3, feature_names=feature_names, class_names=class_names,
          filled=False, rounded=True, fontsize=9)
plt.title('Decision Tree (max_depth=3) — leaf nodes are at the bottom')
plt.show()

print('\nText representation (leaf nodes have no children):')
print(export_text(dt3, feature_names=feature_names))

In [ ]:
# How a leaf node makes a prediction
sample_idx = 0
leaf_id     = dt3.apply(X_test[[sample_idx]])[0]
prediction  = class_names[dt3.predict(X_test[[sample_idx]])[0]]
truth       = class_names[y_test[sample_idx]]

print(f'Test sample features : {X_test[sample_idx]}')
print(f'Arrived at leaf node : {leaf_id}')
print(f'Predicted class      : {prediction}')
print(f'True class           : {truth}')

### ✏️ Exercise 6
Using the breast cancer dataset, sweep `max_depth` from 1 to 10. Print a table of train accuracy, test accuracy, and number of leaf nodes. Identify the best depth.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

# TODO: Sweep max_depth from 1 to 10
# Print: max_depth | train_acc | test_acc | n_leaves
# Identify the best max_depth


---
## 7. Class Imbalance — Why Accuracy Lies

### Concept

When one class dominates (e.g., 99% not-fraud, 1% fraud), a model that **always predicts the majority class** achieves high accuracy but has **zero utility**.

**Better metrics for imbalanced data:**

| Metric | Formula | Use when |
|---|---|---|
| **Precision** | TP / (TP + FP) | False positives are costly (e.g., spam filter flagging real emails) |
| **Recall** | TP / (TP + FN) | False negatives are costly (e.g., missing cancer) |
| **F1** | 2×P×R / (P+R) | Balance both concerns |
| **AUC-ROC** | Area under ROC | Ranking quality across thresholds |

**Techniques to handle imbalance:**
- `class_weight='balanced'` in scikit-learn classifiers
- Oversample minority class (SMOTE)
- Undersample majority class
- Use F1 / AUC instead of accuracy for model selection

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Simulated: 990 non-fraud (0), 10 fraud (1)
y_true = np.array([0]*990 + [1]*10)

# Naive model: always predict 0
y_pred_naive = np.zeros_like(y_true)

print('=== Naive model (always predict 0) ===')
print(f'Accuracy:  {accuracy_score(y_true, y_pred_naive):.4f}  <- looks great!')
print(f'Precision: {precision_score(y_true, y_pred_naive, zero_division=0):.4f}')
print(f'Recall:    {recall_score(y_true, y_pred_naive):.4f}  <- catches 0% of fraud!')
print(f'F1:        {f1_score(y_true, y_pred_naive, zero_division=0):.4f}')
print()

# Better model: catches 8/10 fraud, some false positives
y_pred_better = np.zeros_like(y_true)
y_pred_better[990:998] = 1   # 8 true positives
y_pred_better[5:10]    = 1   # 5 false positives

print('=== Better model (catches fraud, some FP) ===')
print(f'Accuracy:  {accuracy_score(y_true, y_pred_better):.4f}  <- lower accuracy...')
print(f'Precision: {precision_score(y_true, y_pred_better):.4f}')
print(f'Recall:    {recall_score(y_true, y_pred_better):.4f}  <- catches 80% of fraud!')
print(f'F1:        {f1_score(y_true, y_pred_better):.4f}  <- F1 shows the real improvement')

In [ ]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# 95% class 0, 5% class 1
X, y = make_classification(n_samples=1000, weights=[0.95, 0.05],
                            n_features=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

lr_plain    = LogisticRegression(max_iter=1000)
lr_balanced = LogisticRegression(max_iter=1000, class_weight='balanced')

lr_plain.fit(X_train, y_train)
lr_balanced.fit(X_train, y_train)

print('Without class_weight:')
print(classification_report(y_test, lr_plain.predict(X_test)))

print('With class_weight="balanced":')
print(classification_report(y_test, lr_balanced.predict(X_test)))

### ✏️ Exercise 7
Answer in comments: which metric would you prioritize for each scenario, and why?

In [ ]:
# Scenario A: Cancer detection — missing a cancer case (FN) is life-threatening
# → Metric: ???  Reason: ???
# TODO:

# Scenario B: Spam filter — flagging a real email as spam (FP) frustrates users
# → Metric: ???  Reason: ???
# TODO:

# Scenario C: You need to balance both FP and FN equally
# → Metric: ???
# TODO:

---
## 8. ML Pipeline Pattern — Preventing Leakage in Cross-Validation

### Concept

scikit-learn `Pipeline` chains preprocessing and model steps. During `cross_val_score`:

- **Without Pipeline:** If you scale before calling `cross_val_score`, the scaler has already seen all CV folds — leakage.
- **With Pipeline:** The scaler is re-fit on each training fold only. The validation fold is always unseen.

```
Fold 1: [train|train|train|VAL]  → scaler fits on first 3 folds only
Fold 2: [train|train|VAL|train]  → scaler fits on outer 3 folds only
```

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import numpy as np

X, y = load_breast_cancer(return_X_y=True)

# ✅ Correct: scaler inside Pipeline
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(max_iter=1000))
])
scores_correct = cross_val_score(pipe, X, y, cv=5)
print(f'Pipeline CV accuracy:    {scores_correct.mean():.4f} ± {scores_correct.std():.4f}  ✅')

# ❌ Wrong: scaler outside Pipeline
scaler = StandardScaler()
X_prescaled = scaler.fit_transform(X)   # sees ALL CV folds!
lr = LogisticRegression(max_iter=1000)
scores_leaky = cross_val_score(lr, X_prescaled, y, cv=5)
print(f'Pre-scaled CV accuracy:  {scores_leaky.mean():.4f} ± {scores_leaky.std():.4f}  ❌ (leaked)')

### ✏️ Exercise 8
Build a full Pipeline for the Titanic dataset (features: pclass, sex, age → target: survived).  
Use `ColumnTransformer` for mixed types, `KNeighborsClassifier(n_neighbors=7)`, and run 5-fold CV with `scoring='f1'`.

In [ ]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

titanic = pd.read_csv(
    'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv'
)[['pclass', 'sex', 'age', 'survived']].dropna(subset=['survived'])

X = titanic[['pclass', 'sex', 'age']]
y = titanic['survived']

# TODO: Build ColumnTransformer for numeric cols (pclass, age) and categorical (sex)
# TODO: Build Pipeline: ColumnTransformer + KNeighborsClassifier(n_neighbors=7)
# TODO: Run cross_val_score(cv=5, scoring='f1'), print mean ± std


---
## 9. Quiz Results

In [ ]:
quiz_results = {
    'name':           'MinhThuan',
    'date':           '2026-02-27',
    'model_used':     'Claude Sonnet 4.6',
    'overall_score':  13,
    'num_correct':    2,
    'topics_missed': [
        'Supervised vs. Unsupervised Learning',
        'Dependent vs. Independent Variables',
        'Data Leakage / Train-Test Split',
        'Logistic Regression Intercept (log-odds vs. probability)',
        'Cross-Entropy Loss mechanics',
        'KNN — non-parametric, no learned parameters',
        'KNN — k=1 overfitting',
        'Decision Tree Leaf Nodes',
        'Decision Tree max_depth overfitting',
        'Class Imbalance',
        'Pipeline vs. Manual Scaling (CV leakage)',
        'Model selection: regression vs. classification',
        'Curse of Dimensionality',
    ],
    'behavioral_notes':   'Strong on validation set purpose (Q15). Tendency to select the first plausible-sounding option.',
    'next_steps_from_llm': [
        'Complete all TODO exercises in this notebook',
        'Practice explaining each algorithm verbally without code',
        'Memorize the cross-entropy formula; walk through numeric examples by hand',
        'Re-take the quiz after completing exercises',
    ]
}

for k, v in quiz_results.items():
    print(f'{k}: {v}')

---
## 🪞 Reflection

**1. Which 1–2 concepts were most challenging, and why?**

The two hardest concepts for me were **logistic regression** and **KNN overfitting**. For logistic regression, I confused the intercept (β₀) with a probability. I now understand that β₀ is a log-odds value, not a probability — I still need to apply the sigmoid formula to convert it. For KNN, I thought k=1 meant the model was too simple (underfitting), but it is actually the opposite. With k=1, the model memorizes every training point perfectly, which causes overfitting and poor performance on new data.

---

**2. What trade-offs or assumptions did you overlook during the interview?**

I often picked the first answer that sounded correct without checking the other options carefully. For example, I chose "reinforcement learning" for supervised learning without thinking about what "labeled data" means. I also did not think carefully about the difference between log-odds and probability — they are related but not the same thing. Another mistake was thinking that k=1 in KNN is underfitting, when in reality a lower k makes the model more complex and more likely to overfit. In the future, I will read all options before answering and ask myself: "Does this answer fit the exact definition, or just sound similar?"

---

**3. What is your plan to improve over the next week?**

This week I will complete all 8 exercises in this notebook, starting with the cross-entropy function (Exercise 4) because the formula is still not clear in my mind. I will also write short notes in my own words for each algorithm — supervised vs. unsupervised, KNN, logistic regression, and decision trees — without looking at any code. At the end of the week, I will re-take the 15-question quiz to measure my progress. My goal is to score at least 10 out of 15 on the next attempt.